In [1]:
%pip install openai
%pip install pandas
%pip install reportlab
import json

import reportlab
from dotenv import load_dotenv

load_dotenv()
from openai import OpenAI

client = OpenAI()
from pathlib import Path

import pandas as pd
from openai import BadRequestError

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [65]:
def summarize_with_llm_combined_dataset_only():
    """
    Dataset-only version:
    - Dataset is attached as an input file.
    - Model must compute fairness decomposition from dataset.
    Returns: (system_prompt, user_prompt)
    Column roles:
    - Y (outcome variable): <PUT COLUMN NAME>
    - Z (sensitive attribute): <PUT COLUMN NAME>
    - X (predictor features): <PUT COLUMN NAMES OR 'all except Y and Z'>
    - W (control variables, if any): <PUT COLUMN NAMES OR 'none'>

    """

    prompt_path = Path("prompts_onlygpt.txt")
    system_prompt = prompt_path.read_text(encoding="utf-8")


    user_prompt = """
You are given a dataset as a PDF rendering of the dataset (table)."

Column roles:
- X (protected variable): <Sex>, with x0= Female and x1=Male
- Y (outcome attribute): <Admission> y= 'Accepted'
- Z (spurious features): <None>
- W (mediator variables): <Major>


Your task:
1. Analyze the dataset.
2. Compute the fairness decomposition according to Bareimboim and Plecko theory, both general and X-Z specific effects.
3. Produce a structured report.

Output format MUST be:

TEXT:
<plain language report>

LATEX:
<standalone LaTeX document>
"""

    return system_prompt, user_prompt


In [66]:
def generate_report_from_file_id(
    file_id: str,
    client,
    model: str = "gpt-5-mini",
):
    system_prompt, user_prompt = summarize_with_llm_combined_dataset_only()
    print("USER_PROMPT:", user_prompt)
    try:
        resp = client.responses.create(
            model=model,
            instructions=system_prompt,
            tools=[
                {
                    "type": "code_interpreter",
                    "container": {
                        "type": "auto", 
                        "file_ids": [file_id] 
                    },
                },
            ],
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text": user_prompt},
                ],
            }],
        )
    except BadRequestError as e:
        print("BadRequestError:\n", str(e))
        raise

    print("USAGE:", resp.usage)

    full_output = (resp.output_text or "").strip()

    if "LATEX:" not in full_output:
        raise ValueError(
            "Model output did not contain a LATEX section.\n\nOUTPUT:\n"
            + full_output[:1500]
        )

    text_part, latex_part = full_output.split("LATEX:", 1)
    return text_part.replace("TEXT:", "").strip(), latex_part.strip()

In [ ]:

uploaded = client.files.create(
    file=open("df_filtered.txt", "rb"),
    purpose="assistants",
)

text, latex = generate_report_from_file_id(uploaded.id, client)



USER_PROMPT: 
You are given a dataset as a PDF rendering of the dataset (table)."

Column roles:
- X (protected variable): <Sex>, with x0= Female and x1=Male
- Y (outcome attribute): <Admission> y= 'Accepted'
- Z (spurious features): <None>
- W (mediator variables): <Major>


Your task:
1. Analyze the dataset.
2. Compute the fairness decomposition according to Bareimboim and Plecko theory, both general and X-Z specific effects.
3. Produce a structured report.

Output format MUST be:

TEXT:
<plain language report>

LATEX:
<standalone LaTeX document>

USAGE: ResponseUsage(input_tokens=10090, input_tokens_details=InputTokensDetails(cached_tokens=1792), output_tokens=7108, output_tokens_details=OutputTokensDetails(reasoning_tokens=4608), total_tokens=17198)


In [70]:
print(latex)

\documentclass{article}
\usepackage{geometry}
\geometry{margin=1in}
\usepackage{amsmath}
\begin{document}
\section*{Title: "Fairness Decomposition Report"}

\subsection*{Overview of the Fairness Analysis}
This analysis decomposes an observed disparity in admissions by Sex. We compare $X=x0$ (Female) vs $X=x1$ (Male) and the outcome $Y$ is Admission = "Accepted". The mediator considered is the variable \texttt{Major} and there are no confounders ($Z=$ None). We decomposed the overall group disparity into total variation, total causal effect, an indirect effect through \texttt{Major}, a direct effect, and the spurious effect.

\subsection*{Decomposition of Effects}
\begin{itemize}
\item \textbf{Total variation (observed)}: Male acceptance rate = $0.6459$, Female acceptance rate = $0.4137$. Observed total variation = $0.2322$ (Male $-$ Female). Interpretation: Males are accepted about $0.2322$ more often than Females.
\item \textbf{Total causal effect}: estimated total effect $=$ $0.2322$

In [68]:
text

'Title: "Fairness Decomposition Report"\n\nOverview of the Fairness Analysis\nThis analysis decomposes an observed disparity in admissions by Sex. We compare $X=x0$ (Female) vs $X=x1$ (Male) and the outcome Y is Admission = "Accepted". The mediator considered is Major (variable name: Major) and there are no confounders (Z = None). We decomposed the overall group disparity into total variation, a total causal effect, an indirect effect through Major, a direct effect, and the spurious effect (which is absent).\n\nDecomposition of Effects\n- Total variation (observed difference in acceptance rates): Male acceptance rate = 0.6459, Female acceptance rate = 0.4137. Observed total variation = 0.2322 (Male − Female). Interpretation: Males are accepted about 23.22 percentage points more often than Females in the data.\n\n- Total causal effect (ATE from Female → Male estimated from data): total effect = 0.2322. Interpretation: Changing Sex from Female to Male is associated with an increase of ab

In [69]:
latex

'\\documentclass{article}\n\\usepackage{geometry}\n\\geometry{margin=1in}\n\\usepackage{amsmath}\n\\begin{document}\n\\section*{Title: "Fairness Decomposition Report"}\n\n\\subsection*{Overview of the Fairness Analysis}\nThis analysis decomposes an observed disparity in admissions by Sex. We compare $X=x0$ (Female) vs $X=x1$ (Male) and the outcome $Y$ is Admission = "Accepted". The mediator considered is the variable \\texttt{Major} and there are no confounders ($Z=$ None). We decomposed the overall group disparity into total variation, total causal effect, an indirect effect through \\texttt{Major}, a direct effect, and the spurious effect.\n\n\\subsection*{Decomposition of Effects}\n\\begin{itemize}\n\\item \\textbf{Total variation (observed)}: Male acceptance rate = $0.6459$, Female acceptance rate = $0.4137$. Observed total variation = $0.2322$ (Male $-$ Female). Interpretation: Males are accepted about $0.2322$ more often than Females.\n\\item \\textbf{Total causal effect}: estima